# UNICORN Survey — Example Notebook

**Uniform Near-Infrared CatalOgs from Robust imaging**  (Finkelstein et al. in prep)
---

This notebook demonstrates how to work with UNICORN data products using Python. We use **Maisie's Galaxy** (CEERS ID 6613, Finkelstein et al. 2022) as our example — a galaxy at z~11.4, one of the most distant galaxies discovered in the early JWST era.

We will:
1. Read the photometric and photometric redshift catalogs
2. Plot the observed photometry with upper limits where necessary
3. Overplot the best-fit model fluxes
4. Reconstruct and plot the full model SED from the template library
5. Plot the photometric redshift P(z)

### Data files used
| File | Contents |
|---|---|
| `ceers_photom_v*.fits` | Photometric catalog (ext 1) |
| `ceers_photz_v*.fits` | Photo-z catalog: single values (ext 1), model fluxes (ext 2), P(z) (ext 3), P(z) low-z (ext 4) |
| `unicorn_templates_fiducial.fits` | Template library for SED reconstruction |

## 0. Setup

(Install required packages if needed: pip install astropy numpy matplotlib)
Defines the UNICORN color map


In [14]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from astropy.io import fits
from astropy.table import Table
import warnings
warnings.filterwarnings('ignore')

# ── UNICORN color palette ──────────────────────────────────────────────────────
UNICORN_COLORS = [
    '#4a3a8a',  # deep purple
    '#6b51a3',
    '#8e6bb8',
    '#b07cc6',  # lavender (accent)
    '#d48ec9',
    '#ef9fcd',  # pink
    '#ffb3d9',
    '#ffd4eb',  # pale pink
]
unicorn_cmap = LinearSegmentedColormap.from_list('unicorn', UNICORN_COLORS)

# ── Plotting style — white background ─────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    '#4a3a8a',
    'axes.labelcolor':   'black',
    'axes.titlecolor':   'black',
    'xtick.color':       'black',
    'ytick.color':       'black',
    'text.color':        'black',
    'grid.color':        '#d0c8e8',
    'grid.linestyle':    '--',
    'grid.alpha':        0.6,
    'font.family':       'sans-serif',
    'figure.dpi':        200,
})

# ── Utilities ──────────────────────────────────────────────────────────────────
# Version-safe trapezoid integration (np.trapz deprecated in NumPy 2.0)
try:
    trapz = np.trapezoid
except AttributeError:
    trapz = np.trapz

# P(z) extraction helper — handles single-object (1D) and full catalog (2D)
def extract_pz(arr, idx):
    """Extract P(z) for one object from a column that may be 1D or 2D."""
    a = np.array(arr)
    return a[:, idx] if a.ndim == 2 else a

print('Setup complete.')

Setup complete.


## 1. Reading the catalogs

UNICORN data products are standard FITS binary tables, readable with `astropy`. Each file contains multiple extensions:

**Photometry file** (`*_photom_v*.fits`):
- Ext 1: positions, morphology, fluxes in all filters

**Photo-z file** (`*_photz_v*.fits`):
- Ext 1: single-value quantities (z_a, z_m, confidence intervals, integrated P(z), coefficients)
- Ext 2: best-fit model fluxes per filter (nJy)
- Ext 3: full P(z) — ZGRID, PZ, CHI2 (1751 redshift points)
- Ext 4: P(z) restricted to z < 7 — ZGRID_LOWZ, PZ_LOWZ (351 points)

In the next cell, edit the top three lines to be the correct field, version and path to your local copy of the files.

In [15]:
# ── File paths — update to your local paths ───────────────────────────────────
field   = 'ceers'
version = '0.941'
base    = f'//Users/sf8542/Research/UNICORN/{field.upper()}/Results/'  # <-- update this

photom_file    = base + f'{field}_photom_v{version}.fits'
photz_file     = base + f'Photoz/{field}_photz_v{version}.fits'
template_file  = base + 'unicorn_templates_fiducial.fits'

# ── Read photometry ───────────────────────────────────────────────────────────
phot = Table.read(photom_file, hdu=1)
print(f'Photom catalog: {len(phot):,} objects, {len(phot.colnames)} columns')

# ── Read photo-z extensions ───────────────────────────────────────────────────
pz1 = Table.read(photz_file, hdu=1)   # single-value quantities
pz2 = Table.read(photz_file, hdu=2)   # model fluxes
pz3 = Table.read(photz_file, hdu=3)   # P(z)
pz4 = Table.read(photz_file, hdu=4)   # P(z) low-z
print(f'Photo-z ext 1:  {len(pz1):,} objects, {len(pz1.colnames)} columns')
print(f'Photo-z ext 2:  {len(pz2):,} objects (model fluxes), {len(pz2.colnames)} filters')
print(f'Photo-z ext 3:  {len(pz3):,} redshift points (P(z))')
print(f'Photo-z ext 4:  {len(pz4):,} redshift points (P(z) low-z)')

Photom catalog: 123,184 objects, 239 columns
Photo-z ext 1:  123,184 objects, 58 columns
Photo-z ext 2:  123,184 objects (model fluxes), 11 filters
Photo-z ext 3:  1,751 redshift points (P(z))
Photo-z ext 4:  351 redshift points (P(z) low-z)


In [16]:
# ── Select Maisie's Galaxy ─────────────────────────────────────────────────────

# ── Select by ID with guard ───────────────────────────────────────────────────
TARGET_ID = 6613
match_phot = np.where(phot['ID'] == TARGET_ID)[0]
match_pz   = np.where(pz1['ID'] == TARGET_ID)[0]
if len(match_phot) == 0:
    raise ValueError(f'ID {TARGET_ID} not found in photom catalog.')
if len(match_pz) == 0:
    raise ValueError(f'ID {TARGET_ID} not found in photo-z catalog.')
idx_phot = match_phot[0]
idx_pz   = match_pz[0]

# ── Alternatively, select by RA/Dec with guard ────────────────────────────────
#   search_ra,  search_dec      = 214.943146, 52.942440
#   match_radius_arcsec         = 0.2   # arcsec
#   dra  = (phot['RA']  - search_ra)  * np.cos(np.deg2rad(search_dec)) * 3600.0
#   ddec = (phot['DEC'] - search_dec) * 3600.0
#   dist = np.sqrt(dra**2 + ddec**2)
#   radec_matches = np.where(dist < match_radius_arcsec)[0]
#   if len(radec_matches) == 0:
#       raise ValueError(f'No source found within {match_radius_arcsec}" of ({search_ra}, {search_dec}).')
#   idx_phot = radec_matches[0]
#   print(f'Found a match!  Separation = {dist[idx_phot]:.3f}", ID={phot["ID"][idx_phot]}')
#   idx_pz = np.where(pz1['ID'] == phot['ID'][idx_phot])[0]
#   if len(idx_pz) == 0:
#       raise ValueError(f'ID {phot["ID"][idx_phot]} found in photom but not in photo-z catalog.')
#   idx_pz = idx_pz[0]

src_phot = phot[idx_phot]
src_pz1  = pz1[idx_pz]
src_pz2  = pz2[idx_pz]

# P(z) arrays for this object
# extract_pz handles both single-object files (1D) and full catalogs (2D)
zgrid      = np.array(pz3['ZGRID'])
pz_full    = extract_pz(pz3['PZ'],      idx_pz)
zgrid_lowz = np.array(pz4['ZGRID_LOWZ'])
pz_lowz    = extract_pz(pz4['PZ_LOWZ'], idx_pz)

print(f'Object ID:  {src_phot["ID"]}')
print(f'RA, Dec:    {src_phot["RA"]:.6f}, {src_phot["DEC"]:.6f}')
print(f'z_a:        {src_pz1["ZA"]:.4f}')
print(f'z_m:        {src_pz1["ZM"]:.4f}')
print(f'z_68:       [{src_pz1["ZL68"]:.3f}, {src_pz1["ZU68"]:.3f}]')

Object ID:  6613
RA, Dec:    214.943146, 52.942440
z_a:        11.5200
z_m:        11.3400
z_68:       [10.860, 11.720]


## 2. Assembling the photometry

We use the **fiducial fluxes** (`FLUX_*`) — these are the Kron-corrected fluxes with PSF and aperture corrections applied, including the inaccurate-Kron correction where `APERFLAGS != 0`. Uncertainties are empirically derived.

Filters included:
- **ACS**: F435W, F606W, F814W
- **JWST NIRCam**: F090W, F115W, F150W, F200W, F277W, F356W, F410M, F444W, F470N
- *WFC3 (F105W, F125W, F140W, F160W) available but commented out — not used for photo-z fitting*

Fluxes consistent with zero (S/N < 1) are treated as **upper limits** and plotted as downward arrows.

In [ ]:
# ── Filter definitions ─────────────────────────────────────────────────────────
# Pivot wavelengths in Angstrom
FILTER_WAVES = {
    'F435W': 4328,  'F606W': 5921,  'F814W': 8057,
    # WFC3 — available in catalog but not used for photo-z fitting
    # 'F105W': 10552, 'F125W': 12486, 'F140W': 13923, 'F160W': 15369,
    'F090W': 9016,  'F115W': 11544, 'F150W': 15007, 'F200W': 19886,
    'F277W': 27577, 'F356W': 35682, 'F410M': 40822, 'F444W': 44360,
    'F470N': 47063,
}

# Filter color scheme — ACS in deep purple, JWST in lavender to pink
ACS_COLOR  = '#6b51a3'
JWST_COLOR = '#b07cc6'
WFC3_COLOR = '#6b51a3'  # if you uncomment WFC3

def filter_color(fname):
    if fname in ('F435W','F606W','F814W'):
        return ACS_COLOR
    return JWST_COLOR

# ── Extract fluxes ─────────────────────────────────────────────────────────────
waves, fluxes, errors, colors, names = [], [], [], [], []

for filt, wave in FILTER_WAVES.items():
    fcol  = f'FLUX_{filt}'
    ecol  = f'FLUXERR_{filt}'
    if fcol not in src_phot.colnames:
        continue
    f = float(src_phot[fcol])
    e = float(src_phot[ecol])
    # Skip clearly bad/missing data (e.g. error = 1e12 flags no coverage)
    if e > 1e6:
        continue
    waves.append(wave)
    fluxes.append(f)
    errors.append(e)
    colors.append(filter_color(filt))
    names.append(filt)

waves  = np.array(waves)
fluxes = np.array(fluxes)
errors = np.array(errors)

# Detections: S/N >= 1; non-detections: upper limits
snr       = fluxes / errors
detected  = snr >= 1.0
uplim     = ~detected

print(f'Filters with coverage: {len(waves)}')
print(f'Detections (S/N >= 1): {detected.sum()}')
print(f'Upper limits:          {uplim.sum()}')

In [ ]:
# ── Model fluxes (ext 2) ───────────────────────────────────────────────────────
MODEL_WAVES = {
    'F435W': 4328,  'F606W': 5921,  'F814W': 8057,
    'F090W': 9016,  'F115W': 11544, 'F150W': 15007, 'F200W': 19886,
    'F277W': 27577, 'F356W': 35682, 'F410M': 40822, 'F444W': 44360,
}

model_waves  = []
model_fluxes = []
for filt, wave in MODEL_WAVES.items():
    if filt in pz2.colnames:
        mf = float(src_pz2[filt])
        if mf > 0:  # only plot non-zero model fluxes
            model_waves.append(wave)
            model_fluxes.append(mf)

model_waves  = np.array(model_waves)
model_fluxes = np.array(model_fluxes)
print(f'Model flux filters with non-zero values: {len(model_waves)}')

## 3. Reconstructing the model SED from templates

The full model spectrum is reconstructed by combining the Lazy.jl template library with the fitted coefficients (`COEFFS` in photo-z ext 1). The template file contains:
- **Ext 1**: wavelength grid (10,000 points, Angstrom)
- **Ext 2**: redshift grid (same as P(z) zgrid)
- **Ext 3**: template names
- **Ext 4**: template flux cube — shape (n_templates, n_wave, n_z) in nJy

The model at a given redshift is: `model(λ) = Σ coeffs[i] × template[i](λ, z_best)`

In [ ]:
# ── Load template library ──────────────────────────────────────────────────────
with fits.open(template_file) as hdul:
    wave_templ  = np.array(hdul[1].data['WAVE'].flatten())      # (n_wave,) REST-FRAME Angstroms
    zgrid_templ = np.array(hdul[2].data['ZGRID'].flatten())     # (n_z,)
    templ_names = np.array([r['TNAME'].strip() for r in hdul[3].data])
    templ_cube  = np.array(hdul[4].data['FLUX'])                # (n_templ, n_wave, n_z)

print(f'Template wavelength range: {wave_templ[0]:.0f} – {wave_templ[-1]:.0f} Å  (rest-frame)')
print(f'Template z grid:           {zgrid_templ[0]:.3f} – {zgrid_templ[-1]:.3f} ({len(zgrid_templ)} points)')
print(f'Number of templates:       {len(templ_names)}')
print(f'Template names:            {list(templ_names)}')
print(f'Template cube shape:       {templ_cube.shape}  (n_templ, n_wave, n_z)')

In [ ]:
# ── Reconstruct model SED at z_a ──────────────────────────────────────────────
za     = float(src_pz1['ZA'])
coeffs = np.array(src_pz1['COEFFS']).flatten()   # (n_templates,)

# Find nearest redshift in template grid
iz_best = np.argmin(np.abs(zgrid_templ - za))
print(f'z_a = {za:.4f},  nearest template z = {zgrid_templ[iz_best]:.4f}')
print(f'Template cube shape: {templ_cube.shape}')

# mwrfits writes 2D array tags in column-major order, so astropy reads the
# cube as (n_templ, n_z, n_wave). We index the redshift axis (axis 1).
# Verify: axis 1 should have length n_z, axis 2 should have length n_wave.
if templ_cube.shape[1] == len(zgrid_templ):
    # Shape is (n_templ, n_z, n_wave) — correct for mwrfits output
    templates_at_z = templ_cube[:, iz_best, :]    # (n_templ, n_wave)
else:
    # Shape is (n_templ, n_wave, n_z) — index last axis
    templates_at_z = templ_cube[:, :, iz_best]    # (n_templ, n_wave)

# Linear combination: model = coeffs @ templates_at_z  -> shape (n_wave,)
model_sed = coeffs @ templates_at_z
print(f'model_sed shape: {model_sed.shape}  (should be n_wave={templ_cube.shape[-1]})')

# Also reconstruct low-z model
z_lowz      = float(src_pz1['Z_LOWZ'])
coeffs_lowz = np.array(src_pz1['COEFFS_LOWZ']).flatten()
iz_lowz     = np.argmin(np.abs(zgrid_templ - z_lowz))
if templ_cube.shape[1] == len(zgrid_templ):
    templates_lowz = templ_cube[:, iz_lowz, :]
else:
    templates_lowz = templ_cube[:, :, iz_lowz]
model_lowz = coeffs_lowz @ templates_lowz

print(f'Low-z model at z = {z_lowz:.4f}')

## 4. Photometry + SED plot

We plot:
- **Filled circles**: detected photometry (S/N ≥ 1), colored by instrument
- **Downward arrows**: upper limits (3σ)
- **Open squares**: best-fit model fluxes at observed filter wavelengths
- **Solid line**: full reconstructed model SED from templates

In [ ]:
# ── SED + P(z) combined figure ────────────────────────────────────────────────
# SED takes 2/3 of width, P(z) takes 1/3 — matching the UNICORN bioplot layout

FS = 14   # base font size — all text scaled relative to this

fig, (ax, ax_pz) = plt.subplots(1, 2, figsize=(14, 5),
                                  gridspec_kw={'width_ratios': [2, 1]})

wave_um  = waves  / 1e4
mwave_um = model_waves / 1e4

# ── Model SED (full spectrum) ─────────────────────────────────────────────────
# wave_templ is in rest-frame Angstroms — must redshift to observed frame
wave_obs_hz  = wave_templ * (1 + za)     / 1e4   # observed microns, high-z model
wave_obs_lz  = wave_templ * (1 + z_lowz) / 1e4   # observed microns, low-z model

sed_mask    = (wave_obs_hz > 0.3) & (wave_obs_hz < 5.5) & (model_sed   > 1e-3)
sed_mask_lz = (wave_obs_lz > 0.3) & (wave_obs_lz < 5.5) & (model_lowz > 1e-3)

ax.plot(wave_obs_hz[sed_mask],  model_sed[sed_mask],
        color='#4a3a8a', lw=1.2, alpha=0.85, zorder=2, label=f'Best-fit SED (z={za:.2f})')
ax.plot(wave_obs_lz[sed_mask_lz], model_lowz[sed_mask_lz],
        color='#d48ec9', lw=1.0, alpha=0.7, ls='--', zorder=2,
        label=f'Low-z SED (z={z_lowz:.2f})')

# ── Model fluxes at filter positions (filled squares) ────────────────────────
ax.scatter(mwave_um, model_fluxes, marker='s', s=60,
           color='#4a3a8a', zorder=5, label='Model fluxes')

# ── Observed photometry — detections ─────────────────────────────────────────
ACS_COLOR  = '#6b51a3'
JWST_COLOR = '#b07cc6'
for i in range(len(waves)):
    if detected[i]:
        ax.errorbar(wave_um[i], fluxes[i], yerr=errors[i],
                    fmt='o', color=colors[i], ms=7, lw=1.5,
                    capsize=3, capthick=1.5, zorder=6)

# ── Upper limits ──────────────────────────────────────────────────────────────
for i in range(len(waves)):
    if uplim[i]:
        ulim = 3 * errors[i]
        ax.annotate('', xy=(wave_um[i], ulim * 0.55),
                    xytext=(wave_um[i], ulim),
                    arrowprops=dict(arrowstyle='->', color=colors[i],
                                   lw=1.5, mutation_scale=14))
        ax.plot(wave_um[i], ulim, '_', color=colors[i], ms=12, lw=2, zorder=6)

# ── Info box (upper left, outside plot area or inset) ─────────────────────────
# Compute derived quantities
def flux_to_mag(flux_njy):
    """Convert flux in nJy to AB magnitude."""
    if flux_njy > 0:
        return -2.5 * np.log10(flux_njy * 1e-9) - 48.6
    return 99.0

f277 = float(src_phot['FLUX_F277W']) if 'FLUX_F277W' in src_phot.colnames else 0
f444 = float(src_phot['FLUX_F444W']) if 'FLUX_F444W' in src_phot.colnames else 0
rh277 = float(src_phot['RH_F277W'])  if 'RH_F277W'  in src_phot.colnames else 0
rh444 = float(src_phot['RH_F444W'])  if 'RH_F444W'  in src_phot.colnames else 0
m277  = flux_to_mag(f277)
m444  = flux_to_mag(f444)
depthtier = int(src_phot['DEPTHTIER']) if 'DEPTHTIER' in src_phot.colnames else 0
chia  = float(src_pz1['CHIA'])

info_lines = [
    f'{field.upper()} v{version}  |  ID {TARGET_ID}  |  tier{depthtier}',
    f'{float(src_phot["RA"]):.5f}, {float(src_phot["DEC"]):.5f}',
    f'z$_{{best}}$={za:.2f} ($\Delta\chi^2$={chia:.2f})  z$_{{low-z}}$={z_lowz:.2f}',
    f'm$_{{277}}$={m277:.1f}  m$_{{444}}$={m444:.1f}',
    f'r$_{{h,277}}$={rh277:.2f} pix  r$_{{h,444}}$={rh444:.2f} pix',
]
info_text = '\n'.join(info_lines)

ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
        fontsize=FS * 0.78, verticalalignment='top', horizontalalignment='left',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='#4a3a8a', alpha=0.85),
        family='monospace', zorder=10, linespacing=1.4)

# ── Legend (instrument + model labels) ───────────────────────────────────────
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=ACS_COLOR,
           markersize=9, label='HST/ACS'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=JWST_COLOR,
           markersize=9, label='JWST/NIRCam'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#4a3a8a',
           markersize=8, label='Model fluxes'),
    Line2D([0],[0], color='#4a3a8a', lw=1.5, label=f'Best-fit SED (z={za:.2f})'),
    Line2D([0],[0], color='#d48ec9', lw=1.5, ls='--', label=f'Low-z SED (z={z_lowz:.2f})'),
]
ax.legend(handles=legend_elements, fontsize=FS * 0.85, loc='upper right',
          framealpha=0.85, facecolor='white', edgecolor='#4a3a8a')

# ── Axes formatting ───────────────────────────────────────────────────────────
ax.set_xscale('log')
ax.set_xlim(0.35, 5.5)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x:.1f}'))
ax.xaxis.set_minor_formatter(ticker.NullFormatter())
ax.tick_params(axis='both', labelsize=FS)

ymax = np.nanmax(fluxes[detected]) * 2.2 if detected.any() else 50
ymin = -0.2 * ymax
ax.set_ylim(ymin, ymax)
ax.axhline(0, color='#9b80b8', lw=0.8, ls=':')
ax.grid(True, which='major', alpha=0.3)
ax.set_xlabel('Observed Wavelength (μm)', fontsize=FS)
ax.set_ylabel('Flux (nJy)', fontsize=FS)

# Lyman break annotation
lya_obs = 1216 * (1 + za) / 1e4
ax.axvline(lya_obs, color='#8e6bb8', lw=1, ls=':', alpha=0.7)
ax.text(lya_obs * 1.04, ymax * 0.88, 'Lyα', color='#6b51a3', fontsize=FS * 0.85)

# ── P(z) panel ────────────────────────────────────────────────────────────────
pz_norm    = pz_full  / trapz(pz_full,  zgrid)      if pz_full.sum()  > 0 else pz_full
pz_lz_norm = pz_lowz  / trapz(pz_lowz,  zgrid_lowz) if pz_lowz.sum() > 0 else pz_lowz

ax_pz.plot(zgrid, pz_norm, color='#4a3a8a', lw=1.5, zorder=3, label='Fiducial')
ax_pz.fill_between(zgrid, pz_norm, alpha=0.2, color='#4a3a8a', zorder=2)
ax_pz.plot(zgrid_lowz, pz_lz_norm, color='#d48ec9', lw=1.5, ls='--', zorder=3, label='Low-z (z<7)')
ax_pz.axvline(za, color='#6b51a3', lw=1.5, ls=':', zorder=4, label=f'z_a={za:.2f}')

ax_pz.set_xlabel('Redshift', fontsize=FS)
ax_pz.set_ylabel('P(z)  [normalized]', fontsize=FS)
ax_pz.set_xlim(0, 14)
ax_pz.set_ylim(bottom=0)
ax_pz.tick_params(axis='both', labelsize=FS)
ax_pz.legend(fontsize=FS * 0.85, framealpha=0.85, facecolor='white', edgecolor='#4a3a8a')
ax_pz.grid(True, alpha=0.3)

fig.suptitle(f"Maisie's Galaxy — CEERS ID {TARGET_ID} — z_a = {za:.2f}",
             fontsize=FS * 1.2, y=1.01)
plt.tight_layout()
plt.savefig('maisie_sed_pz.pdf', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: maisie_sed_pz.pdf')

## 5. Photometric redshift P(z)

The P(z) is stored as a 2D array in photo-z ext 3 (full range, z=0–35) and ext 4 (low-z restricted, z<7). We plot both on the same panel, along with the best-fit redshift and 68%/95% confidence intervals.

In [ ]:
# P(z) is now plotted alongside the SED in the cell above.
pass

## 6. Useful catalog columns reference

Key columns you may want to use from the photom and photo-z catalogs:

## 5. Column definitions

UNICORN FITS files carry column descriptions and units directly in the header — no separate README needed. The cell below prints the key columns for quick reference. You can also inspect any column interactively with:

```python
phot['RA'].description   # column description
phot['RA'].unit          # physical units
```

In [ ]:
# ── Column definitions from FITS metadata ────────────────────────────────────
# UNICORN catalogs embed column descriptions (TCOMM) and units (TUNIT)
# directly in the FITS headers. This cell reads them out for reference.

print('=' * 75)
print('PHOTOM CATALOG — Key columns')
print('=' * 75)
key_phot = [
    'ID', 'RA', 'DEC', 'DETECTCAT', 'APERFLAGS', 'STARFLAGS', 'STELLARITY',
    'KRON_RADIUS', 'D_APER', 'SCALE_KRON', 'SCALE_MODEL',
    'FLUX_F444W', 'FLUXERR_F444W', 'FLUX_APER_F444W',
    'RH_F444W', 'EDGEFLAGS_51PIX_F444W',
    'NEIGHBOR_D_CLOSEST', 'NEIGHBOR_MAG_CLOSEST',
]
with fits.open(photom_file) as hdul:
    hdr = hdul[1].header
    for col in key_phot:
        for j in range(1, hdr['TFIELDS']+1):
            if hdr.get(f'TTYPE{j}','') == col:
                unit = hdr.get(f'TUNIT{j}', '—')
                comm = hdr.get(f'TCOMM{j}', '')
                print(f'  {col:35s}  [{unit:8s}]  {comm}')

print()
print('=' * 75)
print('PHOTO-Z EXT 1 — Key columns')
print('=' * 75)
key_pz1 = [
    'ID', 'ZA', 'ZM', 'CHIA', 'ZL68', 'ZU68', 'ZL95', 'ZU95',
    'INT_CEN', 'INT_ZGT6', 'INT_ZGT7', 'INT_ZGT8', 'INT_ZGT9',
    'INT_PZ9P5_12', 'SAMPLE_JWST', 'SAMPLE_INTEGER',
    'Z_LOWZ', 'CHIA_LOWZ', 'COEFFS', 'COEFFS_LOWZ',
    'M1500', 'M1300',
]
with fits.open(photz_file) as hdul:
    hdr = hdul[1].header
    for col in key_pz1:
        for j in range(1, hdr['TFIELDS']+1):
            if hdr.get(f'TTYPE{j}','') == col:
                comm = hdr.get(f'TCOMM{j}', '')
                print(f'  {col:35s}  {comm}')

print()
print('=' * 75)
print('PHOTO-Z EXT 2 — Model fluxes')
print('=' * 75)
print('  One column per filter — Lazy.jl best-fit model flux at z_a (nJy)')
with fits.open(photz_file) as hdul:
    hdr = hdul[2].header
    filters = [hdr.get(f'TTYPE{j}','') for j in range(1, hdr['TFIELDS']+1)]
    print(f'  Filters: {filters}')

print()
print('=' * 75)
print('PHOTO-Z EXT 3 — P(z)')
print('=' * 75)
print('  ZGRID    : Redshift grid (1751 points, z=0 to 35)')
print('  PZ       : P(z) probability distribution (normalized to unit integral)')
print('  CHI2     : Chi-squared as a function of redshift')
print()
print('PHOTO-Z EXT 4 — P(z) low-z')
print('  ZGRID_LOWZ : Redshift grid restricted to z < 7 (351 points)')
print('  PZ_LOWZ    : P(z) from low-z restricted run')

---

## Citation

If you use UNICORN data products, please cite:

> Finkelstein et al. (in prep), *The UNICORN Survey*

For Maisie's Galaxy specifically:

> Finkelstein et al. (2022), *ApJL*, 940, L55 — [arXiv:2207.12474](https://arxiv.org/abs/2207.12474)

For the CEERS survey:

> Finkelstein et al. (2025), *ApJL*, 983, 4 — [ADS](https://ui.adsabs.harvard.edu/abs/2025ApJ...983L...4F/abstract)